<a href="https://colab.research.google.com/github/imaisnaini/data-science-2026/blob/main/Pertemuan10_Fatimah_Isnaini_Shabrina_250401020073.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Practice 10 : Algoritma Klasifikasi (Bagian 2)
---
##👩‍🎓 **Student Information**
- **Name**: Fatimah Isnaini Shabrina
- **NIM**: 250401020073
- **University**: UNSIA - Universitas Siber Asia
- **Class**: IF401 Data Science
---
### Langkah 1 : Muat dan Eksplorasi Data

In [22]:
import pandas as pd
from google.colab import drive

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print('Dataset Shape : ', df.shape)
print(df["Churn"].value_counts(normalize=True))
print(df.info)

Dataset Shape :  (7043, 21)
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64
<bound method DataFrame.info of       customerID  gender  SeniorCitizen Partner Dependents  tenure  \
0     7590-VHVEG  Female              0     Yes         No       1   
1     5575-GNVDE    Male              0      No         No      34   
2     3668-QPYBK    Male              0      No         No       2   
3     7795-CFOCW    Male              0      No         No      45   
4     9237-HQITU  Female              0      No         No       2   
...          ...     ...            ...     ...        ...     ...   
7038  6840-RESVB    Male              0     Yes        Yes      24   
7039  2234-XADUH  Female              0     Yes        Yes      72   
7040  4801-JZAZL  Female              0     Yes        Yes      11   
7041  8361-LTMKD    Male              1     Yes         No       4   
7042  3186-AJIEK    Male              0      No         No      66   

     PhoneService     Multiple

### Langkah 2: Preprocessing

In [23]:
from sklearn.model_selection import train_test_split

# Drop ID column (not useful for prediction)
df = df.drop(columns=['customerID'])

# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Encode target (Churn)
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

# TODO: encoding fitur kategorikal
X = pd.get_dummies(df.drop('Churn', axis=1), drop_first=True)

# TODO: pisahkan X (fitur) dan y (target = Churn)
y = df['Churn']

print("Shape X:", X.shape)
print("Shape y:", y.shape)
print(df.info)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

Shape X: (7043, 30)
Shape y: (7043,)
<bound method DataFrame.info of       gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0     Female              0     Yes         No       1           No   
1       Male              0      No         No      34          Yes   
2       Male              0      No         No       2          Yes   
3       Male              0      No         No      45           No   
4     Female              0      No         No       2          Yes   
...      ...            ...     ...        ...     ...          ...   
7038    Male              0     Yes        Yes      24          Yes   
7039  Female              0     Yes        Yes      72          Yes   
7040  Female              0     Yes        Yes      11           No   
7041    Male              1     Yes         No       4          Yes   
7042    Male              0      No         No      66          Yes   

         MultipleLines InternetService OnlineSecurity OnlineBackup  \
0     No

### Langkah 3: Latih Model

In [25]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

### Langkah 4: Evaluasi

In [26]:
from sklearn.metrics import classification_report, roc_auc_score

# TODO: hitung prediksi dan probabilitas
y_pred = rf.predict(X_te)
y_prob = rf.predict_proba(X_te)[:, 1]

# TODO: tampilkan classification_report dan ROC-AUC
print("=== Classification Report ===")
print(classification_report(y_te, y_pred))

print(f"ROC-AUC Score : {roc_auc_score(y_te, y_prob):.4f}")

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC Score : 0.8246


### Langkah 5: Prediksi Probabilitas dan Simpulkan

In [31]:
# TODO: hitung probabilitas churn (predict_proba)
y_prob = rf.predict_proba(X_te)[:, 1]

print("Probabilitas Churn (5 data pertama):")
result = pd.DataFrame({
    "Actual": y_te.values,
    "Prediction": y_pred,
    "Churn Probability": y_prob
})

result.head()

Probabilitas Churn (5 data pertama):


,Actual,Prediction,Churn Probability
0,0,0,0.000000
1,0,1,0.786667
2,0,0,0.090000
3,0,0,0.280000
4,0,0,0.000000


##  📝 Notes :

* Model Random Forest mencapai **akurasi keseluruhan sebesar 79%** dengan **skor ROC-AUC sebesar 0,8246**, yang menunjukkan kemampuan yang baik untuk membedakan antara pelanggan yang kemungkinan akan berhenti berlangganan dan pelanggan yang tidak.
* Model ini memperoleh **presisi sebesar 63%** dan **recall sebesar 50%** untuk kelas pelanggan yang berhenti berlangganan, artinya model ini berhasil mengidentifikasi setengah dari pelanggan yang benar-benar berhenti berlangganan.
* Meskipun model ini berkinerja baik dalam memprediksi pelanggan yang tidak berhenti berlangganan, recall-nya untuk pelanggan yang berhenti berlangganan masih relatif rendah, menunjukkan bahwa beberapa kasus pelanggan yang berpotensi berhenti berlangganan terlewatkan.